# Interview Simulation Agent

## AI Recruitment Intelligence Platform

This notebook implements an AI-powered interview simulation system.

The agent analyzes:

- Candidate Profile
- Job Profile
- ATS Analysis Report
- Career Recommendation Report

and generates:

- Personalized interview configuration
- Technical interview questions
- Candidate answer evaluation
- Follow-up questions
- Final interview performance report


## Architecture

Candidate Data

↓

Interview Configuration Agent

↓

Question Generation Agent

↓

Interview Simulation Agent

↓

Answer Evaluation Agent

↓

Follow-up Question Agent

↓

Performance Analysis Agent

↓

Interview Report

In [2]:
!pip install -q transformers
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 46.6 MB/s eta 0:00:00:00:0100:01


In [3]:
import json
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

os.makedirs(
    "/kaggle/working/outputs",
    exist_ok=True
)

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)


print("Model loaded successfully")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully


In [5]:
def generate_text(prompt):

    messages = [
        {
            "role": "system",
            "content": 
            "You are an AI recruitment agent. Return only valid JSON. No explanations."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)



    outputs = model.generate(

        **inputs,

        max_new_tokens=800,

        temperature=0.1,

        do_sample=False,

        pad_token_id=tokenizer.eos_token_id

    )


    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


    # Remove prompt part
    result = result.split(
        "assistant"
    )[-1]


    return result.strip()

In [6]:
def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)



base_path = "/kaggle/input/datasets/mennaallahwalid/ai-recruitment-outputs"



candidate_profile = load_json(
    f"{base_path}/candidate_profile.json"
)


job_profile = load_json(
    f"{base_path}/job_profile.json"
)


ats_report = load_json(
    f"{base_path}/ats_report.json"
)


career_report = load_json(
    f"{base_path}/career_report.json"
)



print("Interview inputs loaded successfully")

Interview inputs loaded successfully


# Interview Configuration Agent

This agent dynamically creates an interview configuration
for any candidate and any job role.

It does not depend on a specific profession.

The agent analyzes:

- Job requirements
- Candidate background
- ATS results
- Skill gaps

and decides:

- Interview type
- Difficulty level
- Number of questions
- Interview focus areas
- Interview objective

In [7]:
def extract_json(text):

    text = text.strip()

    if "```" in text:
        text = text.replace("```json", "")
        text = text.replace("```", "")

    start = text.find("{")
    end = text.rfind("}") + 1

    text = text[start:end]

    return json.loads(text)

In [8]:
interview_config_prompt = f"""

You are an Interview Configuration Agent
for an AI Recruitment Intelligence Platform.


Your task is to configure a realistic personalized interview
for any job role based on the actual job requirements.


The system must work for different professions including:

- Engineering
- Software Development
- Data Science
- Finance
- Healthcare
- Business
- Marketing
- Administration
- Other professional fields


Analyze ONLY:

1. Candidate Profile
2. Job Profile
3. ATS Report
4. Career Report



====================
OUTPUT RULES
====================

Return ONLY valid JSON.

Do not add markdown.

Do not add explanations.

Follow exactly this JSON structure.

Do not add extra keys.



Required JSON format:


{{
    "interview_type": "",
    "interview_goal": "",
    "difficulty": "",
    "number_of_questions": 0,
    "focus_areas": []
}}



====================
INTERVIEW TYPE RULES
====================


Select the most suitable interview type based on
the Job Profile and role requirements.


Available options:


- Technical Interview
- HR Interview
- Mixed Interview
- Project Discussion
- Domain Knowledge Interview



Rules:


Technical Interview:

Use when the role requires:
- Technical skills
- Engineering knowledge
- Programming
- Tools
- Technical methodologies
- Practical technical tasks



HR Interview:

Use when the role mainly evaluates:
- Communication
- Motivation
- Behavioral skills
- Work attitude
- Culture fit



Mixed Interview:

Use when both technical and behavioral evaluation
are important.



Project Discussion:

Use when the candidate has relevant projects
directly related to the target role.



Domain Knowledge Interview:

Use for specialized professions requiring
specific professional knowledge.



IMPORTANT:

- Do not select Technical Interview by default.
- The interview type must depend on the actual job requirements.
- Choose the type that best matches the real interview process
for this role.



====================
INTERVIEW GOAL RULES
====================


Generate a short realistic interview goal.

The goal must describe what the interviewer wants
to evaluate.


Examples:


Technical role:

"Assess technical knowledge, problem-solving ability, and role-specific competencies."


Non-technical role:

"Assess professional knowledge, experience, and role-related competencies."


Rules:

- Keep it general.
- Do not mention technologies unless they are required
  in the Job Profile.



====================
DIFFICULTY RULES
====================


Determine difficulty using candidate level only.


Junior:

Choose:

- Easy
- Medium


Mid-Level:

Choose:

- Medium


Senior:

Choose:

- Medium
- Hard



IMPORTANT:

- Do not determine difficulty from job title.
- Do not classify based only on number of projects.



====================
NUMBER OF QUESTIONS
====================


Choose:


5:

Short interview.


8:

Standard detailed interview.


10:

Full interview simulation.



====================
FOCUS AREAS RULES
====================


Generate interview focus areas based on the
actual job requirements.


The focus areas represent the topics that a real
interviewer would evaluate during the interview.



Use ONLY:


1. Job responsibilities.

2. Required skills.

3. Preferred skills.

4. Industry/domain requirements.

5. Candidate relevant experience and projects.



IMPORTANT:


- Job Profile is the primary source.
- Focus areas must match the target role.
- Generate different focus areas for different professions.
- Do not use predefined categories.
- Do not assume any specific industry.
- Do not copy all candidate skills.
- Do not include unrelated skills.



Specific technologies, tools, frameworks, or software:


- Include them only if they appear in the Job Profile.
- Do not add technologies only because they exist in the candidate CV.
- Do not invent requirements.



Convert job requirements into realistic interview topics.


Examples:


Requirement:

"Experience with Python, TensorFlow, and ML model deployment"


Focus area:

"Machine Learning development and model deployment"



Requirement:

"Experience with AutoCAD and SolidWorks"


Focus area:

"CAD design and mechanical modeling"



Requirement:

"Experience with financial reporting and Excel"


Focus area:

"Financial analysis and reporting"



Generate between 5 and 10 relevant focus areas.



====================
CANDIDATE PROFILE
====================


{json.dumps(candidate_profile, indent=4)}



====================
ATS REPORT
====================


{json.dumps(ats_report, indent=4)}



====================
CAREER REPORT
====================


{json.dumps(career_report, indent=4)}



====================
JOB PROFILE
====================


{json.dumps(job_profile, indent=4)}

"""

In [9]:
config_response = generate_text(
    interview_config_prompt
)


interview_config = extract_json(
    config_response
)


print(
    json.dumps(
        interview_config,
        indent=4,
        ensure_ascii=False
    )
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
    "interview_type": "Technical Interview",
    "interview_goal": "Assess technical knowledge, problem-solving ability, and role-specific competencies.",
    "difficulty": "Medium",
    "number_of_questions": 8,
    "focus_areas": [
        "Machine Learning algorithms and their applications",
        "Deep Learning frameworks (TensorFlow, PyTorch)",
        "Computer Vision techniques and tools",
        "Natural Language Processing (NLP) methods and libraries",
        "Data preprocessing and feature engineering",
        "Model evaluation metrics and techniques",
        "Deployment of ML models using APIs or Streamlit",
        "SQL databases and ETL operations"
    ]
}


# Testing Interview Configuration Agent

Testing the agent with another job role
to verify that the interview adapts according to the Job Profile.

In [10]:
test_job_profile = {

    "job_title": "Mechanical Design Engineer",

    "required_skills": [
        "SolidWorks",
        "AutoCAD",
        "Mechanical Design",
        "Engineering Drawings",
        "3D Modeling"
    ],

    "preferred_skills": [
        "ANSYS",
        "Simulation",
        "Manufacturing Knowledge"
    ],

    "experience_requirements": "Entry Level",

    "education_requirements": "Mechanical Engineering degree"

}
print(
    json.dumps(
        test_job_profile,
        indent=4
    )
)

{
    "job_title": "Mechanical Design Engineer",
    "required_skills": [
        "SolidWorks",
        "AutoCAD",
        "Mechanical Design",
        "Engineering Drawings",
        "3D Modeling"
    ],
    "preferred_skills": [
        "ANSYS",
        "Simulation",
        "Manufacturing Knowledge"
    ],
    "experience_requirements": "Entry Level",
    "education_requirements": "Mechanical Engineering degree"
}


In [11]:
test_interview_config_prompt = interview_config_prompt.replace(
    json.dumps(job_profile, indent=4),
    json.dumps(test_job_profile, indent=4)
)

In [12]:
test_config_response = generate_text(
    test_interview_config_prompt
)


print(test_config_response)

{
    "interview_type": "Technical Interview",
    "interview_goal": "Assess technical knowledge and practical skills in mechanical design and CAD software.",
    "difficulty": "Medium",
    "number_of_questions": 8,
    "focus_areas": [
        "Mechanical Design Principles",
        "CAD Software Proficiency (SolidWorks, AutoCAD)",
        "3D Modeling Techniques",
        "Engineering Drawings and Standards",
        "Simulation and Manufacturing Processes",
        "Mechanical Components and Assemblies",
        "Design for Manufacturing (DFM) and Assembly",
        "Problem-Solving in Mechanical Design"
    ]
}


In [13]:
test_interview_config = extract_json(
    test_config_response
)


print(
    json.dumps(
        test_interview_config,
        indent=4
    )
)

{
    "interview_type": "Technical Interview",
    "interview_goal": "Assess technical knowledge and practical skills in mechanical design and CAD software.",
    "difficulty": "Medium",
    "number_of_questions": 8,
    "focus_areas": [
        "Mechanical Design Principles",
        "CAD Software Proficiency (SolidWorks, AutoCAD)",
        "3D Modeling Techniques",
        "Engineering Drawings and Standards",
        "Simulation and Manufacturing Processes",
        "Mechanical Components and Assemblies",
        "Design for Manufacturing (DFM) and Assembly",
        "Problem-Solving in Mechanical Design"
    ]
}


# Save Interview Configuration

The generated interview configuration will be saved
and used by the next interview agents.

In [14]:
import os
import json


os.makedirs(
    "outputs",
    exist_ok=True
)


with open(
    "outputs/interview_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        interview_config,
        f,
        indent=4,
        ensure_ascii=False
    )


print("Interview configuration saved successfully")

Interview configuration saved successfully


# Question Generation Agent

This agent generates personalized interview questions
based on:

- Candidate profile
- Job requirements
- ATS analysis
- Interview configuration


The generated questions are role-specific
and designed to simulate a real interview process.

In [15]:
question_generation_prompt = f"""

You are a Question Generation Agent
for an AI Recruitment Intelligence Platform.


Your task is to generate realistic interview questions
for a candidate based on the target job requirements.


Analyze ONLY:

1. Candidate Profile
2. Job Profile
3. ATS Report
4. Interview Configuration



====================
OUTPUT RULES
====================

Return ONLY valid JSON.

Do not add markdown.

Do not add explanations.

Do not add extra keys.



Required JSON format:


{{
    "questions": [
        {{
            "question": "",
            "category": "",
            "difficulty": "",
            "expected_answer_points": []
        }}
    ]
}}



====================
QUESTION GENERATION RULES
====================


Generate questions based on:

- Interview type
- Difficulty level
- Focus areas
- Required job skills
- Candidate background



IMPORTANT:


- Questions must match the actual job role.
- Do not ask about unrelated skills.
- Do not copy questions from the CV.
- Questions should simulate a real interviewer.
- Avoid generic questions.
- Include practical and conceptual questions.



Question categories can include:

- Technical Knowledge
- Practical Experience
- Problem Solving
- Project Discussion
- Domain Knowledge
- Behavioral (if needed)



Difficulty:

Use the interview difficulty:

Easy:
- Basic concepts
- Fundamental understanding


Medium:
- Practical scenarios
- Application questions
- Problem solving


Hard:
- Advanced concepts
- Design decisions
- Complex scenarios



Expected answer points:

Provide key points that a good answer should contain.



Generate exactly:

{interview_config["number_of_questions"]}

questions.



====================
CANDIDATE PROFILE
====================

{json.dumps(candidate_profile, indent=4)}



====================
JOB PROFILE
====================

{json.dumps(job_profile, indent=4)}



====================
ATS REPORT
====================

{json.dumps(ats_report, indent=4)}



====================
INTERVIEW CONFIGURATION
====================

{json.dumps(interview_config, indent=4)}

"""

In [16]:
questions_response = generate_text(
    question_generation_prompt
)


print(questions_response)

{
    "questions": [
        {
            "question": "Can you explain the difference between supervised and unsupervised learning? Provide an example of each and discuss when it would be appropriate to use one over the other.",
            "category": "Machine Learning algorithms and their applications",
            "difficulty": "Medium",
            "expected_answer_points": ["Definition of supervised and unsupervised learning", "Example of each", "When to use one over the other"]
        },
        {
            "question": "How would you design a deep neural network for image classification using TensorFlow? Discuss the architecture, layers, and any considerations you would take into account.",
            "category": "Deep Learning frameworks (TensorFlow, PyTorch)",
            "difficulty": "Medium",
            "expected_answer_points": ["Architecture of the network", "Layers used", "Considerations like data augmentation, transfer learning, and optimization techniques"]
      

In [17]:
questions_report = extract_json(
    questions_response
)


print(
    json.dumps(
        questions_report,
        indent=4
    )
)

{
    "questions": [
        {
            "question": "Can you explain the difference between supervised and unsupervised learning? Provide an example of each and discuss when it would be appropriate to use one over the other.",
            "category": "Machine Learning algorithms and their applications",
            "difficulty": "Medium",
            "expected_answer_points": [
                "Definition of supervised and unsupervised learning",
                "Example of each",
                "When to use one over the other"
            ]
        },
        {
            "question": "How would you design a deep neural network for image classification using TensorFlow? Discuss the architecture, layers, and any considerations you would take into account.",
            "category": "Deep Learning frameworks (TensorFlow, PyTorch)",
            "difficulty": "Medium",
            "expected_answer_points": [
                "Architecture of the network",
                "Layers used",


# Save Generated Interview Questions

The generated questions are saved to be used
by the Interview Simulation and Answer Evaluation agents.

In [18]:
import os
import json


os.makedirs(
    "outputs",
    exist_ok=True
)


with open(
    "outputs/interview_questions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        questions_report,
        f,
        indent=4,
        ensure_ascii=False
    )


print("Interview questions saved successfully")

Interview questions saved successfully


# Interview Simulation Agent

This agent simulates a real interview experience.

It:

- Presents generated interview questions.
- Collects candidate answers.
- Maintains interview conversation history.
- Saves the complete interview session.

The collected answers will be evaluated
by the Answer Evaluation Agent.

In [19]:
import json
import os


def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)



interview_questions = load_json(
    "outputs/interview_questions.json"
)


print(
    f"Number of questions: {len(interview_questions['questions'])}"
)

Number of questions: 8


## Interview Session Initialization

A new interview session is created to store:

- Questions
- Candidate answers
- Categories
- Difficulty levels

In [20]:
interview_session = {

    "interview_type": interview_config["interview_type"],

    "difficulty": interview_config["difficulty"],

    "conversation": []

}



print("Interview session initialized")

Interview session initialized


## Interactive Interview Simulation

The agent asks questions one by one
and records candidate responses.

In [21]:
def run_interview_simulation(questions):

    conversation = []


    for i, item in enumerate(questions, start=1):

        print("\n" + "="*60)

        print(
            f"Question {i}/{len(questions)}"
        )


        print(
            "\n",
            item["question"]
        )


        answer = input(
            "\nYour answer: "
        )


        conversation.append(

        {

            "question_id": i,

            "question": item["question"],

            "category": item["category"],

            "difficulty": item["difficulty"],

            "expected_answer_points": item["expected_answer_points"],

            "answer": answer

        }
    
    )


    return conversation

In [22]:
conversation = run_interview_simulation(

    interview_questions["questions"]

)


interview_session["conversation"] = conversation


print(
    "Interview completed successfully"
)


Question 1/8

 Can you explain the difference between supervised and unsupervised learning? Provide an example of each and discuss when it would be appropriate to use one over the other.



Your answer:  Supervised learning is a machine learning approach where the model is trained using labeled data, meaning that each input has a known output. The model learns the relationship between inputs and outputs to make predictions on new unseen data.  Examples include classification, such as spam detection, and regression, such as predicting house prices.  Unsupervised learning uses unlabeled data and aims to discover hidden patterns or structures within the data. An example is clustering customers based on their behavior.  I would use supervised learning when labeled data is available and the goal is prediction. I would use unsupervised learning when labels are unavailable and we need to explore patterns or relationships in the data.



Question 2/8

 How would you design a deep neural network for image classification using TensorFlow? Discuss the architecture, layers, and any considerations you would take into account.



Your answer:  For image classification, I would design a convolutional neural network using TensorFlow.  The architecture can include convolutional layers for extracting visual features, activation functions such as ReLU, pooling layers for reducing spatial dimensions, and fully connected layers for classification.  If the dataset is limited, I would use transfer learning with pretrained models like ResNet or EfficientNet.  I would also consider data augmentation, choosing an appropriate optimizer, tuning the learning rate, using regularization techniques such as dropout, and evaluating the model using validation data to avoid overfitting.



Question 3/8

 Explain the concept of transfer learning and provide an example of how you might apply it in a computer vision project. How does it differ from fine-tuning?


...


Question 7/8

 How would you deploy a machine learning model using Streamlit? Walk me through the process from creating the UI to making predictions.



Your answer:  To deploy a machine learning model using Streamlit, I would first train and save the model using a format such as pickle or joblib.  Then I would create a Streamlit application that loads the trained model, provides user input components, preprocesses the input data, and sends it to the model for prediction.  Finally, the application displays the prediction results through an interactive user interface.  The same approach can also be integrated with APIs for production deployment.



Question 8/8

 Given a SQL database schema, describe how you would extract relevant features for a machine learning model. What considerations would you take into account when designing the ETL process?



Your answer:  First, I would understand the database structure and identify the relevant tables and columns related to the machine learning problem.  Then I would use SQL queries to extract the required data and perform transformations to create meaningful features.  The ETL process includes extracting data from sources, cleaning and transforming it, handling missing values, and loading the processed dataset for model training.  I would also consider data quality, scalability, and ensuring that the extracted features represent the real-world problem.


Interview completed successfully


# Answer Evaluation Agent

This agent evaluates candidate answers based on:

- Interview question
- Candidate answer
- Expected answer points

The agent generates:

- Score
- Evaluation
- Strengths
- Missing points
- Improvement feedback

In [23]:
def evaluate_answer(item):

    evaluation_prompt = f"""

You are an Answer Evaluation Agent
for an AI Recruitment Intelligence Platform.


Your task is to evaluate a candidate answer
during a realistic interview simulation.


The system supports any job role including:

- Engineering
- Software Development
- Data Science
- Finance
- Healthcare
- Business
- Other professional fields



Analyze ONLY:

1. Interview Question
2. Question Category
3. Candidate Answer
4. Expected Answer Points



====================
OUTPUT RULES
====================

Return ONLY valid JSON.

No markdown.
No explanations outside JSON.

Follow exactly this structure:


{{
    "question_id": {item["question_id"]},
    "score": 0,
    "evaluation": "",
    "strengths": [],
    "missing_points": [],
    "improvement_feedback": ""
}}



====================
SCORING RULES
====================


Score from 0 to 10.


0-3:
Poor answer.
Missing most important concepts.


4-6:
Partially correct.
Some important points are missing.


7-8:
Good answer.
Covers most requirements.


9-10:
Excellent answer.
Complete, accurate, and well explained.



====================
EVALUATION CRITERIA
====================


Evaluate based on:

- Technical correctness
- Completeness
- Relevance to the job role
- Clarity
- Practical understanding



====================
INTERVIEW QUESTION
====================

{item["question"]}



====================
QUESTION CATEGORY
====================

{item["category"]}



====================
CANDIDATE ANSWER
====================

{item["answer"]}



====================
EXPECTED ANSWER POINTS
====================

{json.dumps(
    item.get("expected_answer_points", []),
    indent=4
)}


"""


    response = generate_text(
        evaluation_prompt
    )


    evaluation = extract_json(response)


    return evaluation

In [24]:
answer_evaluations = []


for item in conversation:

    evaluation = evaluate_answer(item)

    answer_evaluations.append(
        evaluation
    )


print("Answer evaluation completed successfully")


print(
    json.dumps(
        answer_evaluations,
        indent=4
    )
)

Answer evaluation completed successfully
[
    {
        "question_id": 1,
        "score": 8,
        "evaluation": "The candidate provided a clear and concise explanation of supervised and unsupervised learning, along with relevant examples and a discussion on when to use each. The answer is complete and relevant to the job role.",
        "strengths": [
            "Correct definitions of supervised and unsupervised learning",
            "Provided relevant examples for each type",
            "Discussed appropriate use cases for both types"
        ],
        "missing_points": [],
        "improvement_feedback": ""
    },
    {
        "question_id": 2,
        "score": 8,
        "evaluation": "The candidate provided a good overview of the key components of a CNN for image classification and mentioned important considerations. However, the answer could be more detailed and technical.",
        "strengths": [
            "Correctly identified the use of convolutional layers, activa

In [25]:
import os
import json


os.makedirs(
    "outputs",
    exist_ok=True
)


# Save answer evaluations

with open(
    "outputs/answer_evaluations.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        answer_evaluations,
        f,
        indent=4,
        ensure_ascii=False
    )


print(
    "Answer evaluations saved successfully"
)



# Add evaluations to interview session

interview_session["evaluations"] = answer_evaluations



# Save complete interview session

with open(
    "outputs/interview_session.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        interview_session,
        f,
        indent=4,
        ensure_ascii=False
    )


print(
    "Complete interview session saved successfully"
)

Answer evaluations saved successfully
Complete interview session saved successfully


# Interview Report Generation Agent

The Interview Report Agent analyzes the complete interview session,
including candidate answers and answer evaluations, to generate a
comprehensive interview performance report.

The agent provides:

- Overall interview performance score
- Candidate strengths
- Technical weaknesses
- Missing skills
- Improvement recommendations
- Hiring recommendation

The generated report helps recruiters make data-driven hiring decisions.

In [26]:
interview_report_prompt = f"""

You are an Interview Report Agent
for an AI Recruitment Intelligence Platform.


Your task is to generate a professional final interview
performance report based on the complete interview results.


The platform supports interviews for all professional fields including:

- Engineering
- Software Development
- Data Science
- Finance
- Healthcare
- Business
- Marketing
- Administration
- Other professional roles



Your analysis must use ONLY:

1. Interview Configuration
2. Job Profile
3. Interview Conversation
4. Answer Evaluations



====================
OUTPUT RULES
====================


Return ONLY valid JSON.

No markdown.

No explanations outside JSON.

Do not add extra keys.



Required JSON format:


{{
    "overall_score": 0,
    "performance_level": "",
    "summary": "",
    "strengths": [],
    "weaknesses": [],
    "missing_skills": [],
    "improvement_recommendations": [],
    "hiring_recommendation": ""
}}



====================
OVERALL SCORE
====================


Calculate:

overall_score = average of all answer evaluation scores.


Rules:

- Use ONLY the provided evaluation scores.
- Do not modify the scores.
- Do not increase or decrease manually.
- Score must be between 0 and 10.
- Round to one decimal place.



====================
PERFORMANCE LEVEL
====================


Determine ONLY from overall_score:


If overall_score >= 9.0:

"Excellent"


If overall_score >= 7.0 and < 9.0:

"Good"


If overall_score >= 5.0 and < 7.0:

"Average"


If overall_score < 5.0:

"Needs Improvement"



Important:

The performance_level MUST exactly match
the overall_score range.



====================
SUMMARY
====================


Generate a concise professional summary.


Include:

- Overall interview performance.
- Candidate knowledge level.
- Problem-solving ability.
- Practical understanding.
- Suitability for the target role.


Rules:

- Adapt the summary to the target job.
- Keep it professional.
- Mention technologies, tools, frameworks, or domain concepts ONLY if they appear in:
    - Job Profile
    - Interview Configuration
    - Interview Questions



====================
STRENGTHS
====================


Extract strengths ONLY from:

- High scoring answers.
- Correct answers.
- Strong explanations.
- Relevant practical examples.
- Good problem-solving approaches.


Rules:

- Use evidence from interview answers only.
- Do not copy CV information.
- Do not add generic personality traits.
- Keep strengths related to the target role.



====================
WEAKNESSES
====================


Identify weaknesses ONLY from:

- Low evaluation scores.
- Missing expected answer points.
- Incorrect concepts.
- Incomplete explanations.
- Lack of practical depth.


Rules:

- Every weakness must be supported by Answer Evaluations.
- Do not invent weaknesses.
- Do not assume missing knowledge.
- Keep weaknesses related to evaluated interview topics.



====================
MISSING SKILLS
====================


Compare:

1. Required skills from Job Profile.
2. Candidate interview performance.


Add a skill ONLY if ALL conditions are satisfied:

- The skill exists in Job Profile requirements.
- The skill was evaluated during the interview.
- Candidate demonstrated weakness or insufficient knowledge in that skill.


Important:

- Do NOT convert weaknesses into skills automatically.
- Do NOT add recommendations as skills.
- Do NOT add unrelated technologies.
- Do NOT add general improvement areas.
- If no missing required skills are identified, return [].



====================
IMPROVEMENT RECOMMENDATIONS
====================


Generate actionable recommendations.


Recommendations must be:

- Specific.
- Practical.
- Related to the target role.
- Based only on identified weaknesses.


Avoid:

- Generic advice.
- Unrelated learning suggestions.
- Skills that are not required by the job.



====================
HIRING RECOMMENDATION
====================


Choose ONLY one:


"Strong Hire":

Use when:

- Candidate strongly matches job requirements.
- Interview performance is excellent.
- Few or no important gaps exist.



"Hire":

Use when:

- Candidate matches most requirements.
- Minor improvement areas exist.



"Consider":

Use when:

- Candidate shows potential.
- Noticeable knowledge gaps exist.
- Additional evaluation may be useful.



"Reject":

Use when:

- Candidate does not meet important job requirements.
- Performance is insufficient.



Base hiring recommendation on:

- Overall score.
- Job requirements.
- Interview difficulty.
- Candidate performance.
- Skill match.



====================
INTERVIEW CONFIGURATION
====================

{json.dumps(interview_config, indent=4)}



====================
JOB PROFILE
====================

{json.dumps(job_profile, indent=4)}



====================
INTERVIEW CONVERSATION
====================

{json.dumps(conversation, indent=4)}



====================
ANSWER EVALUATIONS
====================

{json.dumps(answer_evaluations, indent=4)}

"""

In [27]:
interview_report_response = generate_text(
    interview_report_prompt
)


print(interview_report_response)

{
    "overall_score": 7.5,
    "performance_level": "Good",
    "summary": "The candidate demonstrated a solid understanding of machine learning concepts and processes, with a strong grasp of deep learning, NLP, and data preprocessing. However, there are areas for improvement, particularly in providing more detailed technical descriptions and specific examples.",
    "strengths": [
        "Correct definitions of supervised and unsupervised learning",
        "Provided relevant examples for each type",
        "Discussed appropriate use cases for both types",
        "Identified common text preprocessing steps",
        "Mentioned tokenization, stemming, and lemmatization",
        "Recognized the importance of preprocessing in improving model performance",
        "Described general methods for handling missing values and outliers",
        "Discussed accuracy, precision, recall, and F1 score",
        "Provided a clear and comprehensive explanation of deploying a machine learning mo

In [28]:
interview_report = extract_json(
    interview_report_response
)


print(
    json.dumps(
        interview_report,
        indent=4
    )
)

{
    "overall_score": 7.5,
    "performance_level": "Good",
    "summary": "The candidate demonstrated a solid understanding of machine learning concepts and processes, with a strong grasp of deep learning, NLP, and data preprocessing. However, there are areas for improvement, particularly in providing more detailed technical descriptions and specific examples.",
    "strengths": [
        "Correct definitions of supervised and unsupervised learning",
        "Provided relevant examples for each type",
        "Discussed appropriate use cases for both types",
        "Identified common text preprocessing steps",
        "Mentioned tokenization, stemming, and lemmatization",
        "Recognized the importance of preprocessing in improving model performance",
        "Described general methods for handling missing values and outliers",
        "Discussed accuracy, precision, recall, and F1 score",
        "Provided a clear and comprehensive explanation of deploying a machine learning mo

In [29]:
import os
import json


os.makedirs(
    "outputs",
    exist_ok=True
)


with open(
    "outputs/interview_report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        interview_report,
        f,
        indent=4,
        ensure_ascii=False
    )


print("Interview report saved successfully")

Interview report saved successfully


# Final Recruitment Decision Agent

The Final Recruitment Decision Agent is responsible for making the final hiring recommendation by combining multiple recruitment signals.

It analyzes:

- Candidate profile
- Job requirements
- ATS evaluation
- Interview performance report


The agent provides:

- Final hiring decision
- Overall candidate score
- Decision explanation
- Candidate strengths
- Potential risks
- Hiring confidence


This agent helps HR teams make data-driven recruitment decisions instead of relying on a single evaluation source.

In [30]:
final_decision_prompt = f"""

You are a Final Recruitment Decision Agent
for an AI Recruitment Intelligence Platform.


Your task is to make the final hiring decision
based on all available recruitment evaluation results.


The platform supports all professional roles including:

- Engineering
- Software Development
- Data Science
- Finance
- Healthcare
- Business
- Marketing
- Administration
- Other professional fields



Analyze ONLY:

1. Candidate Profile
2. Job Profile
3. ATS Report
4. Interview Report



====================
OUTPUT RULES
====================


Return ONLY valid JSON.

No markdown.

No explanations outside JSON.

Do not add extra keys.



Required JSON format:


{{
    "final_decision": "",
    "overall_score": 0,
    "confidence_level": "",
    "decision_reason": "",
    "candidate_strengths": [],
    "candidate_risks": [],
    "skill_match_summary": "",
    "recommendation_for_hr": ""
}}



====================
FINAL DECISION RULES
====================


Choose ONLY one:


"Strong Hire":

Use when:

- Candidate strongly matches job requirements.
- ATS evaluation is high.
- Interview performance is excellent.
- Required skills are clearly demonstrated.
- No significant risks exist.



"Hire":

Use when:

- Candidate matches most job requirements.
- Interview performance is good.
- Minor gaps exist.
- Candidate is suitable for the role.



"Consider":

Use when:

- Candidate shows potential.
- Some important gaps exist.
- Additional evaluation or training may be required.



"Reject":

Use when:

- Candidate does not meet important role requirements.
- Required skills are missing.
- Interview performance is insufficient.



====================
OVERALL SCORE
====================


Calculate:

overall_score = final candidate score out of 100


Consider ONLY:

- ATS performance.
- Interview performance.
- Skill alignment with Job Profile.


Rules:

- Score must be between 0 and 100.
- Do not manually increase or decrease scores.
- Use evidence from provided reports only.



====================
CONFIDENCE LEVEL
====================


Choose ONLY one:


"High":

When:

- ATS and interview results strongly agree.
- Enough evidence exists for a clear decision.



"Medium":

When:

- Candidate is suitable but some uncertainty exists.
- Some skill gaps or missing evidence exist.



"Low":

When:

- Limited evaluation evidence is available.



====================
DECISION REASON
====================


Generate a professional explanation.

Include:

- Candidate suitability for the role.
- Main supporting evidence.
- Important gaps if they exist.


Rules:

- Use ONLY information from provided inputs.
- Do not invent candidate experience.
- Do not add unsupported assumptions.



====================
CANDIDATE STRENGTHS
====================


Extract strengths from:

- ATS Report.
- Interview Report.
- Candidate Profile.


Rules:

- Use only evidence from provided data.
- Do not copy CV skills directly.
- Keep strengths related to the target role.
- Avoid generic statements.



====================
CANDIDATE RISKS
====================


Identify hiring risks from:

- ATS Report.
- Interview Report.
- Job Profile.


Rules:

- Mention ONLY confirmed risks.
- Every risk must have evidence in the provided reports.
- Do not invent missing skills.
- Do not convert improvement recommendations into risks.
- Do not mention technologies, tools, or practices unless they appear in the Job Profile.
- If no confirmed risks exist, return [].



====================
SKILL MATCH SUMMARY
====================


Summarize:

- Required skills matched.
- Required skills with weak evidence.
- Important skill gaps.


Rules:

- Mention ONLY skills explicitly required by the Job Profile.
- Do not add unrelated technologies.
- Do not include general improvement areas as skills.
- Do not convert recommendations into missing skills.
- If no skill gaps exist, return an empty gap statement.



====================
HR RECOMMENDATION
====================


Provide a practical recommendation
for the recruiter.


Include:

- Suggested hiring action.
- Possible next steps if required.


Rules:

- Keep it professional.
- Base it ONLY on evaluation results.
- Do not suggest unrelated training.



====================
CANDIDATE PROFILE
====================

{json.dumps(candidate_profile, indent=4)}



====================
JOB PROFILE
====================

{json.dumps(job_profile, indent=4)}



====================
ATS REPORT
====================

{json.dumps(ats_report, indent=4)}



====================
INTERVIEW REPORT
====================

{json.dumps(interview_report, indent=4)}

"""

In [31]:
final_decision_response = generate_text(
    final_decision_prompt
)


final_decision = extract_json(
    final_decision_response
)


print(
    json.dumps(
        final_decision,
        indent=4
    )
)

{
    "final_decision": "Hire",
    "overall_score": 75,
    "confidence_level": "Medium",
    "decision_reason": "The candidate demonstrates a solid understanding of machine learning concepts and processes, with a strong grasp of deep learning, NLP, and data preprocessing. The ATS score is above average, and the interview performance is good. However, there are minor gaps in the candidate's knowledge of SQL databases and some areas for improvement in providing more detailed technical descriptions and specific examples. The candidate has relevant experience and skills that align well with the job requirements, making them a suitable fit for the role.",
    "candidate_strengths": [
        "Solid understanding of machine learning concepts and processes",
        "Strong grasp of deep learning, NLP, and data preprocessing",
        "Correct definitions of supervised and unsupervised learning",
        "Provided relevant examples for each type",
        "Discussed appropriate use cases fo

In [32]:
with open(
    "outputs/final_recruitment_decision.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_decision,
        f,
        indent=4,
        ensure_ascii=False
    )


print("Final recruitment decision saved successfully")

Final recruitment decision saved successfully
